# Imports

In [ ]:
! pip install pymongo
import pymongo
from pymongo import UpdateOne
import pandas as pd

from google.colab import userdata
from bson.dbref import DBRef

from sklearn.model_selection import GridSearchCV

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 313.6/313.6 kB 17.8 MB/s eta 0:00:00


# Connecting to MongoDB

In [ ]:
client = pymongo.MongoClient(userdata.get('MONGO_CONN_STR'), connectTimeoutMS=6000000)

try:
  client.admin.command('ping')
  print("Pinged your deployment. You successfully connected to MongoDB!")
except Exception as e:
  print('Failed to connect: ' + e)

Pinged your deployment. You successfully connected to MongoDB!


## Setting up database

In [ ]:
db = client['spotifyDatabase']

tracks = db['tracks']
genres = db['genres']
artists = db['artists']

Data aquisition

In [ ]:
df = pd.read_csv("hf://datasets/maharshipandya/spotify-tracks-dataset/dataset.csv")

df


,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,...,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.7150,87.917,4,acoustic
1,1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,...,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.2670,77.489,4,acoustic
2,2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,...,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.1200,76.332,4,acoustic
3,3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,...,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.1430,181.740,3,acoustic
4,4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,...,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.1670,119.949,4,acoustic
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
113995,113995,2C3TZjDRiAzdyViavDJ217,Rainy Lullaby,#mindfulness - Soft Rain for Mindful Meditatio...,Sleep My Little Boy,21,384999,False,0.172,0.2350,...,-16.393,1,0.0422,0.6400,0.928000,0.0863,0.0339,125.995,5,world-music
113996,113996,1hIz5L4IB9hN3WRYPOCGPw,Rainy Lullaby,#mindfulness - Soft Rain for Mindful Meditatio...,Water Into Light,22,385000,False,0.174,0.1170,...,-18.318,0,0.0401,0.9940,0.976000,0.1050,0.0350,85.239,4,world-music
113997,113997,6x8ZfSoqDjuNa5SVP5QjvX,Cesária Evora,Best Of,Miss Perfumado,22,271466,False,0.629,0.3290,...,-10.895,0,0.0420,0.8670,0.000000,0.0839,0.7430,132.378,4,world-music
113998,113998,2e6sXL2bYv4bSz6VTdnfLs,Michael W. Smith,Change Your World,Friends,41,283893,False,0.587,0.5060,...,-10.889,1,0.0297,0.3810,0.000000,0.2700,0.4130,135.960,4,world-music


Data cleaning

In [ ]:
initial_rows = len(df)
df = df.dropna() # drop row with nan values
rows_removed = initial_rows - len(df)
print(f"{rows_removed} rows removed.")

if 'Unnamed: 0' in df.columns: #drop unnamed/empty column
  df = df.drop(columns=['Unnamed: 0'])
  print("Unnamed column removed successfully.")
else:
  print("No unnamed column found in the DataFrame.")


1 rows removed.
Unnamed column removed successfully.


Creating dataframe of individual artists

In [ ]:

artists_df = df['artists'].str.split(';').explode().reset_index(drop=True)

# Remove duplicates and convert to DataFrame
artists_df = pd.DataFrame(artists_df.unique(), columns=['artist'])

# Display the resulting DataFrame
artists_df


,artist
0,Gen Hoshino
1,Ben Woodward
2,Ingrid Michaelson
3,ZAYN
4,Kina Grannis
...,...
29854,The WRLDFMS Tony Williams
29855,John Wilds
29856,Molly Skaggs
29857,Cuencos Tibetanos Sonidos Relajantes


# Database injection

- insert artists and map their id to their string name
- insert genres and map their id to their string name
- parse artists from tracks and replace their strings with id references
- same thing for genres
- insert the tracks to the database

In [ ]:
# Create a dictionary to map artist names to their corresponding MongoDB IDs


# Extract unique artist names
artist_names = df['artists'].str.split(';').explode().unique()

# Create a list of artist documents for bulk insertion
artist_docs = [{'artist': artist_name} for artist_name in artist_names]

# Use the 'unordered' bulk write operation for better performance
try:
    result = artists.insert_many(artist_docs, ordered=False)
    print(f"Inserted {len(result.inserted_ids)} artists.")
except pymongo.errors.BulkWriteError as bwe:
    # Handle duplicate key errors (if any) during bulk insert
    print(bwe.details)

# Create a dictionary to map artist names to their corresponding MongoDB IDs
artist_name_to_id = {doc['artist']: doc['_id'] for doc in artists.find({}, {'artist': 1, '_id': 1})}

genres_df = df['track_genre'].unique()
genre_name_to_id = {}

for genre_name in genres_df:
    if genre_name not in genre_name_to_id:
      result = genres.insert_one({'genre': genre_name})
      genre_name_to_id[genre_name] = result.inserted_id

# Preprocess the DataFrame
def preprocess_data(row):
    artist_refs = []
    for artist_name in row['artists'].split(';'):
      if artist_name in artist_name_to_id:
        artist_refs.append(DBRef('artists', artist_name_to_id[artist_name]))
      else:
        # Handle cases where artist is not found
        print(f"Warning: Artist '{artist_name}' not found in the database.")

    genre_ref = DBRef('genres', genre_name_to_id[row['track_genre']])
    if genre_ref is None:
        print(f"Warning: Genre '{row['track_genre']}' not found in the database.")

    track_data = {
        'track_id': row['track_id'],
        'artists': artist_refs,
        'track_genre': genre_ref,
    }

    for column in df.columns:
        if column not in ['track_id', 'artists', 'track_genre']:
            track_data[column] = row[column]

    return track_data

preprocessed_data = df.apply(preprocess_data, axis=1).to_list()

# Insert the preprocessed data into the 'tracks' collection

for i in range(0, len(preprocessed_data), 1000):
  batch = preprocessed_data[i:i+1000]
  tracks.insert_many(batch)
  print(f"Inserted {len(batch)} tracks. Progress: {i}/{len(preprocessed_data)}")

# Prepare bulk write operations for artists
artist_update_operations = []
for track in preprocessed_data:
    for artist_ref in track['artists']:
        artist_update_operations.append(
            UpdateOne({'_id': artist_ref.id}, {'$push': {'tracks': DBRef('tracks', track['track_id'])}})
        )

# Execute bulk write for artists
if artist_update_operations:
    artists.bulk_write(artist_update_operations, ordered=False)

# Prepare bulk write operations for genres
genre_update_operations = []
for track in preprocessed_data:
    genre_update_operations.append(
        UpdateOne({'_id': track['track_genre'].id}, {'$push': {'tracks': DBRef('tracks', track['track_id'])}})
    )

# Execute bulk write for genres
if genre_update_operations:
    genres.bulk_write(genre_update_operations, ordered=False)

Inserted 29859 artists.
Inserted 1000 tracks. Progress: 0/113999
Inserted 1000 tracks. Progress: 1000/113999
Inserted 1000 tracks. Progress: 2000/113999
Inserted 1000 tracks. Progress: 3000/113999
Inserted 1000 tracks. Progress: 4000/113999
Inserted 1000 tracks. Progress: 5000/113999
Inserted 1000 tracks. Progress: 6000/113999
Inserted 1000 tracks. Progress: 7000/113999
Inserted 1000 tracks. Progress: 8000/113999
Inserted 1000 tracks. Progress: 9000/113999
Inserted 1000 tracks. Progress: 10000/113999
Inserted 1000 tracks. Progress: 11000/113999
Inserted 1000 tracks. Progress: 12000/113999
Inserted 1000 tracks. Progress: 13000/113999
Inserted 1000 tracks. Progress: 14000/113999
Inserted 1000 tracks. Progress: 15000/113999
Inserted 1000 tracks. Progress: 16000/113999
Inserted 1000 tracks. Progress: 17000/113999
Inserted 1000 tracks. Progress: 18000/113999
Inserted 1000 tracks. Progress: 19000/113999
Inserted 1000 tracks. Progress: 20000/113999
Inserted 1000 tracks. Progress: 21000/113999

# Query Testing

- first 10 artists
- first 10 genres

In [ ]:
# Query the first 10 rows of the artists collection
results = artists.find().limit(10)

# Iterate through the results and print information about each artist
print("Artists:")
for artist in results:
  print(artist)

print('Genres:')
for genre in genres.find().limit(10):
  print(genre)

print('Tracks:')
for track in tracks.find().limit(10):
  print(track)


Artists:
{'_id': ObjectId('67e20950ed477bfd55a0ad6f'), 'artist': 'Gen Hoshino', 'tracks': [DBRef('tracks', '5SuOikwiRyPMVoIQDJUgSV'), DBRef('tracks', '4nmjL1mUKOAfAbo9QG9tSE'), DBRef('tracks', '12qmPGMrOCogibc7qyxT9s'), DBRef('tracks', '3dPpQeLTWjCjEbSevDMQfW'), DBRef('tracks', '2pcuXnZhTirLXsfXGVFTv2'), DBRef('tracks', '7rIBp3U5Igzn44l7Z7mOtE'), DBRef('tracks', '6hDBkm6B8HF9B4oATW28YN'), DBRef('tracks', '5SuOikwiRyPMVoIQDJUgSV'), DBRef('tracks', '5SuOikwiRyPMVoIQDJUgSV'), DBRef('tracks', '5SuOikwiRyPMVoIQDJUgSV')]}
{'_id': ObjectId('67e20950ed477bfd55a0ad70'), 'artist': 'Ben Woodward', 'tracks': [DBRef('tracks', '4qPNDBW1i3p13qLCt0Ki3A'), DBRef('tracks', '1pG5nd6gmfbMwUfT5shDQe'), DBRef('tracks', '7bhHLZxkRekrNPPkEdDTbn'), DBRef('tracks', '14BMBNRzv24eG6OKoIgPfP'), DBRef('tracks', '0Pi3Ua6fJV1Yx5MGXhfybT'), DBRef('tracks', '1wSZdWFZphxDh6iJhWlIUi'), DBRef('tracks', '4uCVtXasfPk1Z0JwrwTIzH'), DBRef('tracks', '2wdfI1pjsfvviE7WTKGSYM'), DBRef('tracks', '1m3Lsbhkn6yL8apzsCiukd'), DBRef('t

# Random halving CV for best parameters for random forest

Do not run this. The best found parameters are implemented in the random forest below.

In [ ]:
from sklearn.experimental import enable_halving_search_cv  # Needed for HalvingRandomSearchCV
from sklearn.model_selection import HalvingRandomSearchCV
from sklearn.ensemble import RandomForestClassifier
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
# Fetch data from MongoDB
genre_data = list(genres.aggregate([
    {
        '$lookup': {
            'from': 'tracks',
            'localField': '_id',
            'foreignField': 'track_genre.$id',
            'as': 'tracks'
        }
    },
    {
        '$unwind': '$tracks'
    },
    {
        '$project': {
            '_id': 0,
            'genre': '$genre',
            'track_id': '$tracks.track_id',
            'popularity': '$tracks.popularity',
            'danceability': '$tracks.danceability',
            'energy': '$tracks.energy',
            'loudness': '$tracks.loudness',
            'speechiness': '$tracks.speechiness',
            'acousticness': '$tracks.acousticness',
            'instrumentalness': '$tracks.instrumentalness',
            'liveness': '$tracks.liveness',
            'valence': '$tracks.valence',
            'tempo': '$tracks.tempo',
        }
    }
]))

# Create a DataFrame from the retrieved data
df_genre_tracks = pd.DataFrame(genre_data)

# Prepare data for model training
X = df_genre_tracks[['danceability', 'energy', 'loudness', 'speechiness',
                     'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']]
y = df_genre_tracks['genre']

# Encode genre labels
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)

# Define the parameter distributions to sample from
param_dist = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['auto', 'sqrt', 'log2']
}

# Create a HalvingRandomSearchCV object

search = HalvingRandomSearchCV(
    estimator=RandomForestClassifier(),
    param_distributions=param_dist,
    n_candidates='exhaust',  # Explore all candidates initially
    factor=3,  # Each iteration reduces candidates by this factor
    resource='n_samples',  # Resource used to control iterations (samples here)
    max_resources=len(X_train),  # Maximum resource value
    scoring='accuracy',
    random_state=42,
    cv=5,  # Cross-validation folds
    verbose=2  # Increased verbosity for monitoring
)

# Fit the model
search.fit(X_train, y_train)

# Get the best parameters
best_params = search.best_params_
print("Best parameters:", best_params)

# Use the best model
best_model = search.best_estimator_

# Make predictions on the test set
y_pred = best_model.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)


# Function to predict the genre of a track
def predict_track_genre(track_features):

    input_features = pd.DataFrame([track_features])
    prediction = model.predict(input_features)[0]
    predicted_genre = le.inverse_transform([prediction])[0]
    return predicted_genre

n_iterations: 4
n_required_iterations: 4
n_possible_iterations: 4
min_resources_: 1140
max_resources_: 91199
aggressive_elimination: False
factor: 3
----------
iter: 0
n_candidates: 79
n_resources: 1140
Fitting 5 folds for each of 79 candidates, totalling 395 fits
[CV] END max_depth=None, max_features=auto, min_samples_leaf=4, min_samples_split=10, n_estimators=100; total time=   0.0s
[CV] END max_depth=None, max_features=auto, min_samples_leaf=4, min_samples_split=10, n_estimators=100; total time=   0.0s
[CV] END max_depth=None, max_features=auto, min_samples_leaf=4, min_samples_split=10, n_estimators=100; total time=   0.0s
[CV] END max_depth=None, max_features=auto, min_samples_leaf=4, min_samples_split=10, n_estimators=100; total time=   0.0s
[CV] END max_depth=None, max_features=auto, min_samples_leaf=4, min_samples_split=10, n_estimators=100; total time=   0.0s
[CV] END max_depth=None, max_features=auto, min_samples_leaf=1, min_samples_split=10, n_estimators=100; total time=   0.

/usr/local/lib/python3.11/dist-packages/sklearn/model_selection/_validation.py:528: FitFailedWarning: 
125 fits failed out of a total of 395.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
125 fits failed with the following error:
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/sklearn/model_selection/_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/usr/local/lib/python3.11/dist-packages/sklearn/base.py", line 1382, in wrapper
    estimator._validate_params()
  File "/usr/local/lib/python3.11/dist-packages/sklearn/base.py", line 436, in _validate_params
    validate_parameter_constraints(
  File "/usr/local/lib/python3.11/dist-packages/sklearn/util

[CV] END max_depth=10, max_features=sqrt, min_samples_leaf=1, min_samples_split=10, n_estimators=200; total time=   3.8s
[CV] END max_depth=10, max_features=sqrt, min_samples_leaf=1, min_samples_split=10, n_estimators=200; total time=   3.3s
[CV] END max_depth=10, max_features=sqrt, min_samples_leaf=1, min_samples_split=10, n_estimators=200; total time=   3.4s
[CV] END max_depth=10, max_features=sqrt, min_samples_leaf=1, min_samples_split=10, n_estimators=200; total time=   4.1s
[CV] END max_depth=10, max_features=sqrt, min_samples_leaf=1, min_samples_split=10, n_estimators=200; total time=   3.4s
[CV] END max_depth=20, max_features=log2, min_samples_leaf=4, min_samples_split=10, n_estimators=100; total time=   2.1s
[CV] END max_depth=20, max_features=log2, min_samples_leaf=4, min_samples_split=10, n_estimators=100; total time=   2.0s
[CV] END max_depth=20, max_features=log2, min_samples_leaf=4, min_samples_split=10, n_estimators=100; total time=   2.4s
[CV] END max_depth=20, max_featu

/usr/local/lib/python3.11/dist-packages/sklearn/model_selection/_search.py:1108: UserWarning: One or more of the test scores are non-finite: [       nan        nan 0.09130536 0.09481413 0.08778886        nan
 0.08954711 0.09656851        nan 0.09393693 0.0895587  0.09481026
 0.09480253        nan        nan 0.09656851 0.0833913         nan
 0.08603061 0.09481026 0.09306361        nan 0.09216323 0.09831517
        nan        nan        nan 0.09042816        nan 0.09042816
 0.09832676 0.10095834        nan 0.08866605        nan 0.0895587
        nan 0.09657238 0.09393693 0.09218641 0.10007342 0.09217482
 0.09128603        nan 0.08953938 0.08954324        nan 0.09217868
        nan 0.09218255 0.09481026 0.09130922 0.09304428        nan
 0.09305588        nan        nan        nan 0.09919236 0.09920396
 0.08691939        nan 0.09394466 0.09041657 0.09656465 0.09306361
 0.08779272 0.09217868 0.10096221        nan 0.09744184 0.09744957
 0.09043203 0.09043203 0.08691166 0.08603061 0.08778113 

[CV] END max_depth=20, max_features=log2, min_samples_leaf=4, min_samples_split=10, n_estimators=100; total time=   6.1s
[CV] END max_depth=20, max_features=log2, min_samples_leaf=4, min_samples_split=10, n_estimators=100; total time=   6.6s
[CV] END max_depth=20, max_features=log2, min_samples_leaf=4, min_samples_split=10, n_estimators=100; total time=   6.4s
[CV] END max_depth=20, max_features=log2, min_samples_leaf=4, min_samples_split=10, n_estimators=100; total time=   6.4s
[CV] END max_depth=20, max_features=log2, min_samples_leaf=4, min_samples_split=10, n_estimators=100; total time=   6.9s
[CV] END max_depth=20, max_features=log2, min_samples_leaf=2, min_samples_split=10, n_estimators=200; total time=  14.0s
[CV] END max_depth=20, max_features=log2, min_samples_leaf=2, min_samples_split=10, n_estimators=200; total time=  13.9s
[CV] END max_depth=20, max_features=log2, min_samples_leaf=2, min_samples_split=10, n_estimators=200; total time=  13.9s
[CV] END max_depth=20, max_featu

/usr/local/lib/python3.11/dist-packages/sklearn/model_selection/_search.py:1108: UserWarning: One or more of the test scores are non-finite: [       nan        nan 0.09130536 0.09481413 0.08778886        nan
 0.08954711 0.09656851        nan 0.09393693 0.0895587  0.09481026
 0.09480253        nan        nan 0.09656851 0.0833913         nan
 0.08603061 0.09481026 0.09306361        nan 0.09216323 0.09831517
        nan        nan        nan 0.09042816        nan 0.09042816
 0.09832676 0.10095834        nan 0.08866605        nan 0.0895587
        nan 0.09657238 0.09393693 0.09218641 0.10007342 0.09217482
 0.09128603        nan 0.08953938 0.08954324        nan 0.09217868
        nan 0.09218255 0.09481026 0.09130922 0.09304428        nan
 0.09305588        nan        nan        nan 0.09919236 0.09920396
 0.08691939        nan 0.09394466 0.09041657 0.09656465 0.09306361
 0.08779272 0.09217868 0.10096221        nan 0.09744184 0.09744957
 0.09043203 0.09043203 0.08691166 0.08603061 0.08778113 

[CV] END max_depth=20, max_features=sqrt, min_samples_leaf=1, min_samples_split=10, n_estimators=200; total time=  42.2s
[CV] END max_depth=20, max_features=sqrt, min_samples_leaf=1, min_samples_split=10, n_estimators=200; total time=  42.9s
[CV] END max_depth=20, max_features=sqrt, min_samples_leaf=1, min_samples_split=10, n_estimators=200; total time=  42.2s
[CV] END max_depth=20, max_features=sqrt, min_samples_leaf=1, min_samples_split=10, n_estimators=200; total time=  42.8s
[CV] END max_depth=20, max_features=sqrt, min_samples_leaf=1, min_samples_split=10, n_estimators=200; total time=  42.2s
[CV] END max_depth=20, max_features=log2, min_samples_leaf=1, min_samples_split=10, n_estimators=300; total time= 1.1min
[CV] END max_depth=20, max_features=log2, min_samples_leaf=1, min_samples_split=10, n_estimators=300; total time= 1.1min
[CV] END max_depth=20, max_features=log2, min_samples_leaf=1, min_samples_split=10, n_estimators=300; total time= 1.1min
[CV] END max_depth=20, max_featu

/usr/local/lib/python3.11/dist-packages/sklearn/model_selection/_search.py:1108: UserWarning: One or more of the test scores are non-finite: [       nan        nan 0.09130536 0.09481413 0.08778886        nan
 0.08954711 0.09656851        nan 0.09393693 0.0895587  0.09481026
 0.09480253        nan        nan 0.09656851 0.0833913         nan
 0.08603061 0.09481026 0.09306361        nan 0.09216323 0.09831517
        nan        nan        nan 0.09042816        nan 0.09042816
 0.09832676 0.10095834        nan 0.08866605        nan 0.0895587
        nan 0.09657238 0.09393693 0.09218641 0.10007342 0.09217482
 0.09128603        nan 0.08953938 0.08954324        nan 0.09217868
        nan 0.09218255 0.09481026 0.09130922 0.09304428        nan
 0.09305588        nan        nan        nan 0.09919236 0.09920396
 0.08691939        nan 0.09394466 0.09041657 0.09656465 0.09306361
 0.08779272 0.09217868 0.10096221        nan 0.09744184 0.09744957
 0.09043203 0.09043203 0.08691166 0.08603061 0.08778113 

Best parameters: {'n_estimators': 300, 'min_samples_split': 10, 'min_samples_leaf': 1, 'max_features': 'log2', 'max_depth': 20}


declare the prediction method with the resultant random forest from the CV

In [ ]:
# Make predictions on the test set
y_pred = best_model.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)


# Function to predict the genre of a track
def predict_track_genre(track_features):

    input_features = pd.DataFrame([track_features])
    prediction = best_model.predict(input_features)[0]
    predicted_genre = le.inverse_transform([prediction])[0]
    return predicted_genre

Accuracy: 0.23258771929824562


create a random forest without the CV (use this one)

In [ ]:

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Fetch data from MongoDB
genre_data = list(genres.aggregate([
    {
        '$lookup': {
            'from': 'tracks',
            'localField': '_id',
            'foreignField': 'track_genre.$id',
            'as': 'tracks'
        }
    },
    {
        '$unwind': '$tracks'
    },
    {
        '$project': {
            '_id': 0,
            'genre': '$genre',
            'track_id': '$tracks.track_id',
            'popularity': '$tracks.popularity',
            'danceability': '$tracks.danceability',
            'energy': '$tracks.energy',
            'loudness': '$tracks.loudness',
            'speechiness': '$tracks.speechiness',
            'acousticness': '$tracks.acousticness',
            'instrumentalness': '$tracks.instrumentalness',
            'liveness': '$tracks.liveness',
            'valence': '$tracks.valence',
            'tempo': '$tracks.tempo',
        }
    }
]))

# Create a DataFrame from the retrieved data
df_genre_tracks = pd.DataFrame(genre_data)

# Prepare data for model training
X = df_genre_tracks[['danceability', 'energy', 'loudness', 'speechiness',
                     'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']]
y = df_genre_tracks['genre']

# Encode genre labels
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)

# Train a Random Forest classifier
model = RandomForestClassifier(
    n_estimators=300,
    max_depth=20,
    min_samples_split=10,
    min_samples_leaf=1,
    max_features='log2',
    random_state=42
)
model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = model.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)


# Function to predict the genre of a track
def predict_track_genre(track_features):

    input_features = pd.DataFrame([track_features])
    prediction = model.predict(input_features)[0]
    predicted_genre = le.inverse_transform([prediction])[0]
    return predicted_genre


track_features = {

    'danceability': 0.7,
    'energy': 0.8,
    'loudness': -5,
    'speechiness': 0.1,
    'acousticness': 0.2,
    'instrumentalness': 0.0,
    'liveness': 0.3,
    'valence': 0.9,
    'tempo': 120,
}

predicted_genre = predict_track_genre(track_features)
print("Predicted Genre:", predicted_genre)



Accuracy: 0.23105263157894737
Predicted Genre: party


prediction tersting area

In [ ]:
humble = {
    'danceability': 0.539,
    'energy': 0.362,
    'loudness': -13.618,
    'speechiness': 0.0313,
    'acousticness': 0.945,
    'instrumentalness': 0.724,
    'liveness': 0.269,
    'valence': 0.587,
    'tempo': 96.286,
}

print(predict_track_genre(humble))

tango


In [ ]:
def get_track_info(track):
  if track:
    print("Song Name:", track['track_name'])

    artist_ref = track['artists'][0]
    artist = artists.find_one({'_id': artist_ref.id})
    if artist:
      print("Artist Name:", artist['artist'])
    else:
      print("Artist not found.")

    track_features = {
        'danceability': track['danceability'],
        'energy': track['energy'],
        'loudness': track['loudness'],
        'speechiness': track['speechiness'],
        'acousticness': track['acousticness'],
        'instrumentalness': track['instrumentalness'],
        'liveness': track['liveness'],
        'valence': track['valence'],
        'tempo': track['tempo'],
    }

    predicted_genre = predict_track_genre(track_features)
    print("Predicted Genre:", predicted_genre)

    genre_ref = track['track_genre']
    genre = genres.find_one({'_id': genre_ref.id})
    if genre:
      print("Actual Genre:", genre['genre'])
    else:
      print("Genre not found.")

  else:
    print("Track not found.")


song_name = input("Enter the song name: ")
track2 = tracks.find_one({'track_name': song_name})
if track2:
  get_track_info(track2)
else:
  print("Track not found.")

Enter the song name: Higher
Song Name: Higher
Artist Name: Jor'dan Armstrong
Predicted Genre: afrobeat
Actual Genre: afrobeat


In [ ]:
def recommend_tracks(track_name, num_recommendations=5):


  # Find the track in the database based on the track name
  input_track = tracks.find_one({'track_name': track_name})

  if not input_track:
    print(f"Track '{track_name}' not found in the database.")
    return []

  # Extract the track's features
  input_track_features = {
      'danceability': input_track['danceability'],
      'energy': input_track['energy'],
      'loudness': input_track['loudness'],
      'speechiness': input_track['speechiness'],
      'acousticness': input_track['acousticness'],
      'instrumentalness': input_track['instrumentalness'],
      'liveness': input_track['liveness'],
      'valence': input_track['valence'],
      'tempo': input_track['tempo'],
  }

  # Predict the genre of the input track
  predicted_genre = predict_track_genre(input_track_features)

  # Find tracks with similar features and genre
  similar_tracks = list(tracks.find({
      'track_genre.$id': input_track['track_genre'].id,
      'track_name': {'$ne': track_name}  # Exclude the input track itself
  }))


  # Sort tracks by similarity (e.g., Euclidean distance of features)
  def calculate_similarity(track):
    track_features = {
        'danceability': track['danceability'],
        'energy': track['energy'],
        'loudness': track['loudness'],
        'speechiness': track['speechiness'],
        'acousticness': track['acousticness'],
        'instrumentalness': track['instrumentalness'],
        'liveness': track['liveness'],
        'valence': track['valence'],
        'tempo': track['tempo'],
    }
    similarity = 0
    for feature in input_track_features:
      similarity += abs(input_track_features[feature] - track_features[feature])
    return similarity

  similar_tracks.sort(key=calculate_similarity)

  return_list = []

  for track in similar_tracks[:num_recommendations]:
    track_artist_ref = track['artists'][0]
    artist = artists.find_one({'_id': track_artist_ref.id})
    if artist:
      return_list.append(track['track_name'] + ' - ' + artist['artist'])
    else:
      print("Artist not found.")

  return return_list

song_name = input("Enter the song name: ")
recommendations = recommend_tracks(song_name)

if recommendations:
    print("Recommended tracks:")
    for track_name in recommendations:
        print("- ", track_name)
else:
    print("No recommendations found.")


Enter the song name: Man in the Box
Recommended tracks:
-  One Headlight - The Wallflowers
-  One Headlight - The Wallflowers
-  El Arte del Buen Comer - Patricio Rey y sus Redonditos de Ricota
-  Passion - RAC
-  A Change Of Heart - The 1975


In [ ]:
!pip install supertree

import matplotlib.pyplot as plt
from supertree import SuperTree

# Create the SuperTree visualization
st = SuperTree(model, X_train, y_train)
st.show_tree(which_tree=0)